This script processes Eurostat animal density data and aggregates it to your custom GeoClusters. It begins by loading the Eurostat-style TSV file estat_tgs00045.tsv and splitting the metadata-packed first column into separate fields (freq, animals, unit, geo_code), followed by basic column name cleaning. It then identifies all year columns from 2016–2024, converts them to numeric, and computes a row-wise mean (avg_2016_2024) representing the average animal density (or population metric) over this period. A two-letter country_code is extracted from geo_code, and the script groups by country_code to sum these averages into a country-level total_density, while excluding Turkey (TR) and Malta (MT). A lookup table maps country codes to full country names, which are merged into the dataset. Each country is then assigned to one of five GeoClusters using a predefined country→GeoCluster mapping, and this cluster information is merged. Finally, the script aggregates total_density (renamed to Animal_Population) by GeoCluster and exports the resulting GeoCluster-level animal population table as a tab-separated file (GeoCluster_Animal_Population.tsv) for downstream analyses.

In [ ]:
import pandas as pd

# STEP 1: Load Eurostat-style TSV file
file_path = "estat_tgs00045.tsv"  # Change this to your file path
df = pd.read_csv(file_path, sep='\t')

# STEP 2: Split metadata column (first column)
df[['freq', 'animals', 'unit', 'geo_code']] = df.iloc[:, 0].str.split(',', expand=True)

# STEP 3: Clean column names
df.columns = df.columns.str.strip()

# STEP 4: Identify and clean year columns (2016–2024)
year_cols = [col.strip() for col in df.columns if col.strip().isdigit() and 2016 <= int(col.strip()) <= 2024]
for col in year_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# STEP 5: Calculate row-wise average
df['avg_2016_2024'] = df[year_cols].mean(axis=1, skipna=True)

# STEP 6: Extract country_code (first two letters of geo_code)
df['country_code'] = df['geo_code'].str[:2]

# STEP 7: Group by country_code and sum
country_sum = df.groupby('country_code', as_index=False)['avg_2016_2024'].sum()
country_sum.rename(columns={'avg_2016_2024': 'total_density'}, inplace=True)

# STEP 8: Remove TR and MT
country_sum = country_sum[~country_sum['country_code'].isin(['TR', 'MT'])]

# STEP 9: Country lookup
country_lookup = pd.DataFrame([
    ("AT", "Austria"), ("BE", "Belgium"), ("BG", "Bulgaria"), ("CH", "Switzerland"),
    ("CY", "Cyprus"), ("CZ", "Czechia"), ("DE", "Germany"), ("DK", "Denmark"),
    ("EE", "Estonia"), ("EL", "Greece"), ("ES", "Spain"), ("FI", "Finland"),
    ("FR", "France"), ("HR", "Croatia"), ("HU", "Hungary"), ("IE", "Ireland"),
    ("IS", "Iceland"), ("IT", "Italy"), ("LT", "Lithuania"), ("LU", "Luxembourg"),
    ("LV", "Latvia"), ("ME", "Montenegro"), ("MK", "North Macedonia"),
    ("NL", "Netherlands"), ("PL", "Poland"), ("PT", "Portugal"), ("RO", "Romania"),
    ("RS", "Serbia"), ("SE", "Sweden"), ("SI", "Slovenia"), ("SK", "Slovakia"),
    ("UK", "United Kingdom"), ("XK", "Kosovo")
], columns=["country_code", "Country"])

# STEP 10: Merge with country names
merged_df = country_sum.merge(country_lookup, on="country_code", how="left")

# STEP 11: GeoCluster map
cluster_map = {
    "GeoCluster_One": [
        "Albania", "Bulgaria", "Cyprus", "Greece", "Kosovo", "North Macedonia", "Serbia", "Romania"
    ],
    "GeoCluster_Two": [
        "Hungary", "Slovakia", "Poland", "Ukraine", "Moldova", "Belarus"
    ],
    "GeoCluster_Three": [
        "Denmark", "France", "Belgium", "Germany", "Iceland", "Ireland", "Luxembourg",
        "Netherlands", "Portugal", "Spain", "Switzerland", "United Kingdom"
    ],
    "GeoCluster_Four": [
        "Estonia", "Finland", "Latvia", "Lithuania", "Norway", "Sweden"
    ],
    "GeoCluster_Five": [
        "Austria", "Montenegro", "Bosnia and Herz.", "Croatia", "Czechia", "Italy", "Slovenia"
    ]
}

# STEP 12: Reverse lookup (country → cluster)
geo_cluster_lookup = []
for cluster, countries in cluster_map.items():
    for country in countries:
        geo_cluster_lookup.append((country, cluster))
geo_cluster_df = pd.DataFrame(geo_cluster_lookup, columns=["Country", "GeoCluster"])

# STEP 13: Merge GeoCluster
merged_with_cluster = merged_df.merge(geo_cluster_df, on="Country", how="left")

# STEP 14: Summarize by GeoCluster
cluster_summary = merged_with_cluster.groupby('GeoCluster', as_index=False)['total_density'].sum()
cluster_summary.rename(columns={'total_density': 'Animal_Population'}, inplace=True)

# STEP 15: Export to TSV
cluster_summary.to_csv("GeoCluster_Animal_Population.tsv", sep='\t', index=False)
